In [14]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai_api_key = os.getenv("OPENAI_API_KEY")

In [15]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

#### Buffer Memory
* RunnableWithMessageHistory keeps chat messages in an in-memory buffer for each session and passes them into the prompt.
* This replaces the legacy LLMChain and ConversationBufferMemory APIs, whose imports fail in this project's Python 3.14 environment.
* Use conversation.invoke({"question": "Bonjour !"}, config=config).content. Reuse the same session_id to retain history; choose a new one to start a separate conversation.
* History is not persisted: restarting the kernel or rerunning the setup cell clears it.

In [16]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder, 
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)
from langchain_core.runnables.history import RunnableWithMessageHistory


prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template("You are a nice chatbot having a friendly conversation with the user."),
        # The 'variable_name' should match the memory key used to store the chat history.
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

histories: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in histories:
        histories[session_id] = InMemoryChatMessageHistory()
    return histories[session_id]

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "demo"}}
conversation.invoke({"question": "Bonjour !"}, config=config).content


c:\Users\ange0\Documents\Formations\GenAI\LangChain Latest\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


"Bonjour ! Comment ça va aujourd'hui ? Que puis-je faire pour vous ?"

In [17]:
conversation.invoke({"question": "Je m'appelle Ange et j'ai bougé 33 fois cette semaine."}, config=config).content

"Bonjour Ange ! Wow, 33 fois en une semaine, c'est beaucoup de mouvement ! Vous avez été très occupé. Qu'est-ce qui vous a amené à bouger autant ?"

In [18]:
conversation.invoke({"question": "Si ma moyenne de mouvement de distance était de 10 km, combien de distance aurais-je parcourue cette semaine ?"}, config=config).content

"Si vous avez bougé 33 fois cette semaine et que la moyenne de distance parcourue à chaque fois est de 10 km, alors vous auriez parcouru un total de 330 km (33 déplacements x 10 km). C'est vraiment impressionnant ! Avez-vous fait ces déplacements à pied, en voiture, en vélo ou d'une autre manière ?"

In [19]:
conversation.invoke({"question": "Je m'appelle Ange et j'ai bougé 33 fois cette semaine."}, config=config).content

"C'est étrange, mais c'est pas grave. Avez-vous un emploi du temps très chargé pour bouger autant de fois en une semaine ?"


#### Conversation Buffer Window Memory
* This LCEL version trims the context sent to the model to the last 3 complete question/answer exchanges.
* It avoids the legacy `langchain.memory` and `ConversationChain` imports, which fail in this project's Python 3.14 environment.

In [21]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

window_size = 3
window_prompt = RunnablePassthrough.assign(
    chat_history=lambda inputs: trim_messages(
        inputs["chat_history"],
        max_tokens=2 * window_size,
        token_counter=len,
        strategy="last",
        start_on="human",
        end_on=("human", "ai"),
    )
) | prompt

In [22]:
conversation_window = RunnableWithMessageHistory(
    window_prompt | llm,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

c:\Users\ange0\Documents\Formations\GenAI\LangChain Latest\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [24]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"Je retiens que vous vous appelez Ange et que vous avez effectué 33 déplacements cette semaine, ce qui a totalisé une distance de 330 km. J'ai aussi noté que vous n'avez pas précisé comment vous avez effectué ces déplacements. Est-ce que vous aimez bouger beaucoup ou est-ce que ces déplacements étaient nécessaires pour votre travail ou d'autres activités ?"

In [25]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"Je retiens que vous vous appelez Ange et que vous avez bougé 33 fois cette semaine. C'est plutôt inhabituel, mais peut-être avez-vous une vie très mouvementée ! Est-ce que ces déplacements étaient liés à des activités personnelles ou professionnelles ?"

In [27]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"Je me souviens que vous vous appelez Ange et que vous avez effectué 33 déplacements cette semaine, totalisant une distance de 330 km. C'est beaucoup de mouvements en une seule semaine ! J'espère que ces déplacements étaient agréables et que vous avez pu profiter de chaque moment. Avez-vous aimé explorer de nouveaux endroits ou était-ce plutôt des déplacements routiniers ?"

In [28]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"Je me souviens que vous vous appelez Ange et que vous avez été très actif cette semaine avec 33 déplacements pour un total de 330 km. Cela signifie que vous avez bougé beaucoup et je suis curieux de savoir quelles étaient les raisons de tous ces déplacements. Était-ce pour le travail, les loisirs ou d'autres activités ?"

In [29]:
conversation_window.invoke(
    {"question": "Ma couleur préférée est le vert ?"},
    config=config,
).content

"C'est une belle couleur ! Le vert est souvent associé à la nature, à la fraîcheur et à la sérénité. Qu'est-ce qui vous attire particulièrement dans le vert ? Avez-vous des objets ou vêtements de cette couleur chez vous ?"

In [30]:
conversation_window.invoke(
    {"question": "Mon animal favori est le chat ?"},
    config=config,
).content

'Les chats sont des animaux adorables et pleins de charme ! Leur indépendance, leur douceur et leur agilité en font des compagnons très appréciés. Avez-vous un chat chez vous ou êtes-vous plutôt un amateur de vidéos de chats mignons sur internet ?'

In [32]:
conversation_window.invoke(
    {"question": "Je conduis une Opel Astra"},
    config=config,
).content

"La Opel Astra est un choix populaire et fiable en matière de véhicule. C'est un modèle polyvalent et confortable. Appréciez-vous particulièrement certaines caractéristiques de votre Opel Astra ? Avez-vous déjà fait de beaux trajets ou de belles découvertes au volant de votre voiture ?"

In [33]:
conversation_window.invoke(
    {"question": "Ma ville préférée est Paris"},
    config=config,
).content

"Paris est une ville magnifique et pleine de charme, avec son architecture emblématique, ses musées renommés, sa délicieuse cuisine et son ambiance romantique. Avez-vous une anecdote ou un lieu préféré à Paris que vous aimez particulièrement ? Qu'est-ce qui vous attire le plus dans cette ville ?"

In [34]:
conversation_window.invoke(
    {"question": "Quelles est ma couleur préférée ?"},
    config=config,
).content

"Je ne suis pas capable de connaître votre couleur préférée, mais j'aimerais bien le savoir ! Quelle est votre couleur préférée ? Avez-vous une raison particulière pour aimer cette couleur en particulier ?"

In [35]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"Je retiens que vous conduisez une Opel Astra, que votre ville préférée est Paris, et j'ai appris que vous ne m'avez pas encore parlé de votre couleur préférée. J'ai aussi remarqué que vous êtes ouvert(e) à discuter de différents sujets et que vous semblez apprécier les échanges amicaux. Est-ce que j'ai bien résumé nos derniers échanges ? Y a-t-il quelque chose d'autre que vous aimeriez partager ou discuter ?"

In [36]:
conversation_window.invoke(
    {"question": "Quel est mon nom ?"},
    config=config,
).content

"Malheureusement, je ne connais pas votre nom car nous sommes en train d'avoir une conversation virtuelle. Cependant, je serais ravi de continuer à discuter avec vous et d'apprendre davantage sur vos préférences, vos opinions ou tout autre sujet que vous aimeriez aborder. N'hésitez pas à me poser des questions ou à partager des informations avec moi !"

#### Chat Message History
* Save list of chat messages and then fetch them all.

In [37]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [40]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("hi!")

history.add_ai_message("whats up?")

In [41]:
history.messages

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='whats up?', additional_kwargs={}, response_metadata={})]